# Reweighting and Effective Sample Size

Train a small flow, draw weighted samples, and compute ESS / basin populations.

In [ ]:
import torch
from boltzmann_generators.analysis import rectangular_region
from boltzmann_generators.energies import DoubleWell2D
from boltzmann_generators.flows import FlowModel, GaussianPrior, RealNVP
from boltzmann_generators.services import AnalysisSuite, SamplingEngine
from boltzmann_generators.training import TrainConfig, Trainer

torch.manual_seed(1)
device = "cpu"
energy = DoubleWell2D(a=4.0)
model = FlowModel(GaussianPrior(2), RealNVP(dim=2, num_layers=4, hidden_dim=32, mask="halves"))
Trainer(model, energy, TrainConfig(n_epochs=20, batch_size=64, w_ml=0.0, w_kl=1.0), device=device).fit()

engine = SamplingEngine(model, energy)
x, log_w, _ = engine.sample_with_weights(512, device=device)
ess = engine.effective_sample_size(log_w)
print(f"ESS = {ess:.1f} / {x.shape[0]}")

left = rectangular_region(x_min=-2, x_max=0, y_min=-2, y_max=2)
right = rectangular_region(x_min=0, x_max=2, y_min=-2, y_max=2)
pops = AnalysisSuite(engine).basin_populations(x, {"left": left, "right": right}, log_w=log_w)
print("Basin populations:", pops)